### Introduction

This notebook will introduce you to network analysis using the `igraph` package in R. It covers preparing data in the formats which `igraph` uses, how to produce informative visualisations of networks, and how to access numerical properties of networks.

Most of the code in this notebook was written by Alice Miller of the [Australian Digital Observatory](https://research.qut.edu.au/digitalobservatory/) at QUT.


In [ ]:
## SETUP ####

### Housekeeping ####

# Clear entire workspace
rm(list = ls())

# Don't display large numbers with scientific notation
options(scipen = 999)


### Install and load libraries ####

# Install
#install.packages("tidyverse")
#install.packages("igraph")

# Load (do this every session)
library(tidyverse)
library(igraph)


### Working directory ####

# To run this code correctly you will need to have the working directory set to
# the folder containing the materials for the workshop.

# For SWAN session, the data files will be in the same directory as the notebook file and you are good to go.

# Otherwise:
# Some checks you might like to run:
# getwd() # displays your current working directory
# dir()   # displays the contents of your current working directory

# If you can see tweets.csv and candidates_info.csv listed, you're good to go.
# If not, change your working directory:
# In RStudio > Session > Set Working Directory > Choose Directory >
# Select the folder containing the materials for the workshop > Open
setwd('C:/Users/Simon/Networks')

In [ ]:
#set url for data files
file_url <- "https://g-b917be.0b2e93.03c0.gaccess.io/Workshops/Networks2026.zip"

# Test case: Check if file_url is empty
if (is.null(file_url) || !nzchar(trimws(file_url))) {
  stop("file_url is empty! Please set a valid URL in Cell 2 before running this cell.")
}

# Uses the URL stored in file_url (defined in Cell 2)
data_dir <- "data"
if (!dir.exists(data_dir)) dir.create(data_dir, recursive = TRUE)

# Derive a filename from the URL path, with a fallback
parsed <- tryCatch(utils::URLdecode(basename(sub("\\?.*$", "", file_url))), error = function(e) "")
file_name <- if (nzchar(parsed)) parsed else "download.zip"
zip_path <- file.path(data_dir, file_name)

# Download ZIP
utils::download.file(file_url, destfile = zip_path, mode = "wb", quiet = FALSE)

# Extract ZIP contents
utils::unzip(zip_path, exdir = data_dir)

cat(sprintf("Downloaded to: %s\n", zip_path))
cat(sprintf("Extracted into: %s folder\n", data_dir))

In [ ]:
#set the working directory and check that we have two csv files there
setwd('C:/Users/Simon/Networks/data/Networks2026')
my_files <- list.files(pattern = '.csv$' )
# inspect
my_files

### Data

As in last week's class, we will be working with the twitter data from candidates in the 2019 federal election.

In [ ]:
## IMPORT DATA ####

### Tweets ####

# Read in from .csv
tweets <- read.csv("tweets.csv",
                    header = TRUE)

# Set encoding for tweet text
# Encoding(tweets$text) <- "UTF-8"

# View the data to make sure it looks OK
head(tweets,5)


### Candidate info ####

# Read in from .csv
candidates <- read.csv("candidates_info.csv",
                        header = TRUE)

# View the data to make sure it looks OK
head(candidates, 5)


### What are we aiming for?

In the first part of the class, we established that two data objects are needed to create a network graph:
- a list of uniquely identified nodes (with attributes as needed)
- a list of pairs of node IDs - the edges of the graph.

We have now looked at the basic structure of our data files. Our aim is to make a visualisation which shows the extent to which candidates retweet other candidates from their own party and the extent to which retweeting crosses party lines. 

Which pieces of information from the data files do we need to access to make the input tables which will allow us do this?

### Cleaning data

There are a lot of parties represented in the data and grouping them makes sense - we don't need to worry about the state-based differences in party names and so on. And (of course) we will need a catch-all 'Other' category.

Code note: the following cell (and almost all further down) use a special symbol **%>%**, usually referred to as **pipe**. This symbol indicates that the output of one operation is passed on as the inout for a following operation.

In [ ]:
# Check how many candidates there are per party
candidates %>% 
  group_by(party) %>% 
  summarise(nbr_candidates = n()) %>% 
  arrange(desc(nbr_candidates))

# Group parties together
candidates <- candidates %>% 
  mutate(group = fct_collapse(party,
                              alp = c("Australian Labor Party",
                                      "Australian Labor Party (Northern Territory) Branch",
                                      "Labor"),
                              lnp = c("Liberal",
                                      "Liberal National Party of Queensland",
                                      "The Nationals",
                                      "Country Liberals (NT)"),
                              grn = c("The Greens",
                                      "The Greens (VIC)"),
                              ind = "Independent",
                              oth = c("Australian Democrats",
                                      "FRASER ANNING'S CONSERVATIVE NATIONAL PARTY",
                                      "Katter's Australian Party (KAP)",
                                      "Labour DLP",
                                      "Love Australia or Leave",
                                      "Reason Australia",
                                      "United Australia Party")))

# Check
summary(candidates$group)


### Building the network

The starting point for our edge list is to look only at tweets which were retweeted i.e. which have a value for **retweeteed_user_id**. This is done by filtering out entries which have **na** (null) values for **retweeted_user_id**.

In [ ]:
#### Create an edge list with attribute ####

# Create counts of the number of retweets between users
retweet_edges <- tweets %>% 
  filter(is.na(retweeted_user_id) == FALSE) %>% 
  group_by(author_id, retweeted_user_id) %>% 
  summarise(nbr_retweets = n())

# View the results (number of retweets between users) to make sure it looks OK
head(retweet_edges, 10)

The next code in the next cell uses the fact that **user_id**, **author_id** and **retweeted_user_id** contain items from the same set of values. This allows us to link different data tables on these values - this is the **left_join** operation.

In [ ]:
#### Add candidate username to the edge list ####

# Add username for the author and rename column appropriately
# Remove unrequired columns
retweet_edges <- retweet_edges %>% 
  left_join(candidates, by = c("author_id" = "user_id")) %>% 
  rename("author_username" = username) %>% 
  select(-party, -group)

# Add username for the account being retweeted and rename column appropriately
# Remove unrequired columns
retweet_edges <- retweet_edges %>% 
  left_join(candidates, by = c("retweeted_user_id" = "user_id")) %>% 
  rename("retweeted_username" = username) %>% 
  select(-party, -group)

# Check
head(retweet_edges, 10)

The edge table needs to be cleaned a little for igraph - again, **na** values are filtered out and the order of the columns is adjusted.

In [ ]:
# Re-order columns as per requirements of igraph
# Filter rows with NAs (only keeping retweets between candidates)
retweet_edges <- na.omit(retweet_edges) %>% 
  ungroup() %>% 
  select(author_username,
         retweeted_username,
         nbr_retweets)

# Check
head(retweet_edges, 10)

### Add parties
We waat to be able to differentiate the party groupings in our visualisation. Colour is an obvous way to do this, so we assign colours to the groups and include this in the nodes table (called **candidates**). Colour is specified here using hexadecimal codes; R also can work with a wide range of named colours and also with RGB triples. Again, **left_join** is used to link data in different tables.

In [ ]:
# To colour code the nodes, a variable called 'color' is required
parties <- data.frame(group = c("alp",
                                "lnp",
                                "grn",
                                "ind",
                                "oth"),
                      color = c("#e41a1c",  # red
                                "#377eb8",  # blue
                                "#4daf4a",  # green
                                "#ff7f00",  # orange
                                "#984ea3")) # purple

# Join colour information to existing candidates data frame
candidates <- candidates %>% 
  left_join(parties, by = "group") %>% 
  select(username, party, group, color)

# Check
head(candidates, 10)

### igraph
Now we can move to making the visualisation. The first step is to make a graph object from the edge table and the node table. 

In [ ]:
# Create igraph object
retweet_g <- graph_from_data_frame(d = retweet_edges,
                                   vertices = candidates,
                                   directed = TRUE)


#### Inspect igraph properties ####

retweet_g
# First number is the number of nodes
# Second number is the number of edges

The graph object is now the input for making a visualisation or **plotting**. We'll start with a version which uses default settings for the plot. Note the first line of code in this cell: making plots of complex graphs is not always a deterministic procedure and repeated plots will start from different random points (and therefore may look different) unless we set a seed.

In [ ]:
### Plot the network diagram ####

# Default plot settings
set.seed(2)
plot(retweet_g)
# Doesn't look good!

### Improving the layout
We will apply various changes to make the visualisation more comprehensible. 

In [ ]:
# Change size of node (vertex) labels
# Change colour of labels
set.seed(2)
plot(retweet_g,
     vertex.label.cex = 0.7,
     vertex.label.color = "grey10")
# Better, but the nodes are obscuring the arrows

# Adjust the size of the nodes
set.seed(2)
plot(retweet_g,
     vertex.label.cex = 0.7,
     vertex.label.color = "grey10",
     vertex.size = 7)

# Adjust the size of the arrows
set.seed(2)
plot(retweet_g,
     vertex.label.cex = 0.7,
     vertex.label.color = "grey10",
     vertex.size = 7,
     edge.arrow.size = 0.2)

# Let's remove the isolated nodes
retweet_g <- delete.vertices(retweet_g, degree(retweet_g) == 0)
set.seed(2)
plot(retweet_g,
     vertex.label.cex = 0.7,
     vertex.label.color = "grey10",
     vertex.size = 7,
     edge.arrow.size = 0.2)


### Edge weighting
And finally, we will use the information about the number of retweets between each pair of nodes to change the width of the edges - more retweets will result in a wider edge. By now, it should be easy to see what is the answer to the question from which we started: do candidates retweet other candidates from their party more than they retweet candidates from other parties?

In [ ]:
# First create a vector of edge weights based on number of retweets
weight1 <- E(retweet_g)$nbr_retweets

# Check
summary(as.factor(weight1))

# Standardise the range of weights to be between 0 and 1
# Add 0.1 (weight cannot be 0)
weight1 <- ((weight1 - min(weight1)) / (max(weight1) - min(weight1))) + 0.1

# Plot the graph with the edge thickness determined by standardised weights
set.seed(2)
plot(retweet_g,
     vertex.label.cex = 0.7,
     vertex.label.color = "grey10",
     vertex.size = 7,
     edge.arrow.size = 0.2,
     edge.width = weight1)


### Different layouts
igraph can make plots with different layouts; this cell shows you a few examples of how these look. As we will see shortly, Gephi allows you to play with layouts in a more interactive way.

In [ ]:
# Kamada and Kawai (a type of radial layout)
# Note: Node labels have been removed
plot(retweet_g,
     vertex.label = NA,
     vertex.size = 7,
     edge.arrow.size = 0.2,
     edge.width = weight1,
     layout = layout.kamada.kawai)

# circle
# Note: Node labels have been removed
plot(retweet_g,
     vertex.label= NA,
     vertex.size = 7,
     edge.arrow.size = 0.2,
     edge.width = weight1,
     layout = layout.circle)

# Tree
# Note: Node labels have been removed
plot(retweet_g,
     vertex.label = NA,
     vertex.size = 7,
     edge.arrow.size = 0.2,
     edge.width = weight1,
     layout = layout.reingold.tilford)

# Layout nicely function
# First, save the results of the layout_nicely() function
nice_layout <- layout_nicely(retweet_g)

# Plot the network diagram specifying your saved output
# Note: Node labels have been removed
plot(retweet_g,
     vertex.label = NA,
     vertex.size = 7,
     edge.arrow.size = 0.2,
     edge.width = weight1,
     layout = nice_layout)


### Quantitative properties of networks

In [ ]:
#### Node degree (total) ####

# Extract the node degree (total) from the igraph object and save it
retweet_g_degree <- degree(retweet_g)

# Turn it into an easy to read data frame, sort it in descending order by degree,
# add party information
retweet_g_degree <- data.frame(
  username = names(retweet_g_degree),
  degree = retweet_g_degree,
  row.names = NULL) %>% 
  arrange(desc(degree)) %>% 
  left_join(candidates, by = "username") %>% 
  select(-color)

# View the results
View(retweet_g_degree)


#### Out degree ####

# Extract the out degree from the igraph object and save it
retweet_g_degree_out <- degree(retweet_g, mode = "out")

# Turn it into an easy to read data frame, sort it in descending order by degree,
# add party information
retweet_g_degree_out <- data.frame(
  username = names(retweet_g_degree_out),
  out_degree = retweet_g_degree_out,
  row.names = NULL) %>% 
  arrange(desc(out_degree)) %>% 
  left_join(candidates, by = "username") %>% 
  select(-color)

# View the results
View(retweet_g_degree_out)


#### In degree ####

# Extract the in degree from the igraph object and save it
retweet_g_degree_in <- degree(retweet_g, mode = "in")

# Turn it into an easy to read data frame, sort it in descending order by degree,
# add party information
retweet_g_degree_in <- data.frame(
  username = names(retweet_g_degree_in),
  in_degree = retweet_g_degree_in,
  row.names = NULL) %>% 
  arrange(desc(in_degree)) %>% 
  left_join(candidates, by = "username") %>% 
  select(-color)

# View the results
View(retweet_g_degree_in)


#### Compare node degree (in/out/total) and retweets ####

retweet_g_degree <- retweet_g_degree %>% 
  left_join(retweet_g_degree_in, by = "username") %>% 
  left_join(retweet_g_degree_out, by = "username") %>% 
  select(username, group, degree, in_degree, out_degree) %>% 
  arrange(desc(degree))

# View the results
View(retweet_g_degree)


In [ ]:
# eigenvector centrality
retweet_eigenVector <- sort(eigen_centrality(retweet_g)$vector, decreasing = TRUE)
as.data.frame(retweet_eigenVector[1:20])

In [ ]:
# export edges and nodes as csv files

write.csv(retweet_edges, 'edges.csv')
write.csv(candidates, 'nodes.csv')